# 02 · Standard FunnyBird CBM: controlled discovery of concept backwash

**Report question.** When one FunnyBird part is replaced while body, pose,
camera, and background stay fixed, does the corresponding concept answer
follow the inserted part or remain attached to the old bird?

**Population.** Standard non-RL CBM. This notebook contains no MCBM and no
visibility-aware relabelled model. Seed-level replication is shown where
accepted outputs exist; the fixed-render causal analysis begins with seed 1.

**Claims available here.** FunnyBird's renderer permits a controlled
donor-part replacement. Therefore a validated positive donor response plus
a remaining source preference can establish the CBM backwash event. Proposed
explanations are weaker unless independently manipulated.


## The implemented CBM and the notation used below

For image `i`, the encoder produces one value for every concept slot:

```text
x_i → image encoder f_θ → z_i = (z_i1, …, z_iJ)
                              ├→ concept heads → concept predictions
                              └→ class head   → species prediction
```

The implementation trains with

`L_CBM = L_task + beta × L_concept`.

The class head reads the complete vector `z_i`; it does not read a list of hard
0/1 concept decisions. Each concept head reads its corresponding `z_ij` slot.
The setup cell below verifies empirically whether the saved run used identity
concept heads. Only after that check may `z_ij` be called the raw concept logit.

| Symbol | Meaning |
|---|---|
| `x_i` | image `i` |
| `y_i` | species label |
| `c_ij` | processed 0/1 label for exact concept `j` |
| `z_ij` | raw value in concept slot `j`; primary grounding quantity |
| `p_ij = sigmoid(z_ij)` | bounded probability; used only for thresholded performance |
| `c_hat_ij = 1[z_ij>0]` | predicted concept presence |
| `v_ig` | whether mapped part mask `g` is visible |
| `a_ig` | visible area of mask `g` |

Ordinary accuracy and recall answer whether predictions agree with labels. They
do **not** answer whether the prediction came from the named pixels.


In [ ]:
import os, json, re, glob, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image as DisplayImage

CURATED = Path(os.environ["CURATED_DATA"])
CWD = Path.cwd()
REPO = CWD if (CWD/"analysis").is_dir() else CWD.parent
sys.path.insert(0, str(REPO/"data"/"funnybirds"))
plt.rcParams.update({"figure.dpi": 120, "axes.grid": False})
ORDER = ["tail", "wing", "beak", "foot", "eye"]
COLORS = {"tail":"#6A0DAD", "wing":"#0072B2", "beak":"#E69F00",
          "foot":"#009E73", "eye":"#CC79A7"}

def require(path, command):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}\nProduce it with: {command}")
    return path

swap_candidates = [
    CURATED/"swap_fixed_v3_matched"/"funnybirds-cbm-s1.csv",
    CURATED/"swap_fixed_v2_attempt2"/"funnybirds-cbm-s1.csv",
    CURATED/"swap_fixed_v2"/"funnybirds-cbm-s1.csv",
]
SWAP = next((p for p in swap_candidates if p.exists()), None)
if SWAP is None:
    raise FileNotFoundError("No accepted fixed-render standard-CBM swap CSV")
S = pd.read_csv(SWAP)
if "response_delta" not in S:
    S["response_delta"] = S.margin - (S.z_new_orig - S.z_old_orig)
S["responded_but_source_wins"] = (S.response_delta > 0) & (S.margin < 0)
print("fixed-render input:", SWAP)
print("rows:", len(S), "parts:", sorted(S.part.unique()))

PRED_DIR = REPO/"external"/"minimal_cbm"/"results"/"funnybirds-cbm"/"1"/"predictions"
PRED = require(PRED_DIR/"epoch_100.pth", "train or restore funnybirds-cbm seed 1 epoch 100")
import torch
saved = torch.load(PRED, map_location="cpu", weights_only=False)
z_saved = saved["z"].detach().cpu().numpy().reshape(len(saved["z"]), -1)
p_saved = saved["c_preds"].detach().cpu().numpy().reshape(len(saved["c_preds"]), -1)
c_saved = saved["c"].detach().cpu().numpy().reshape(len(saved["c"]), -1)
logit_p = np.log(np.clip(p_saved, 1e-7, 1-1e-7) / np.clip(1-p_saved, 1e-7, 1))
identity_max_error = float(np.max(np.abs(z_saved-logit_p)))
if identity_max_error > 1e-4:
    raise RuntimeError(f"concept head is not identity: max |z-logit(p)|={identity_max_error}")
print(f"[IDENTITY CONCEPT HEAD PASS] max |z-logit(p)|={identity_max_error:.3g}")

FB_ROOT = Path(os.environ.get("FUNNYBIRDS_ROOT", CURATED/"FunnyBirds"))
import funnybirds_concepts as fbc
parts = fbc.load_parts(FB_ROOT)
CONCEPT_NAMES = fbc.concept_names(parts)
SPANS = fbc.group_slices(parts)
if len(CONCEPT_NAMES) != z_saved.shape[1]:
    raise RuntimeError("parts.json concept width does not match saved predictions")
CONCEPT_PART = {name: part for part,(a,b) in SPANS.items() for name in CONCEPT_NAMES[a:b]}
print("checkpoint:", PRED, "concepts:", len(CONCEPT_NAMES), "species:", len(np.unique(saved["y"])))


## 1 · Did training produce a usable, non-collapsed CBM?

**Question.** Did training produce a usable, non-collapsed CBM?

**Variables and prediction.** For every exact concept `j`, measure raw-score spread, positive-versus-negative label separation, balanced accuracy, and positive recall. A usable slot has nonzero spread, positive label separation, and above-chance thresholded performance.

**Method.** Compute all quantities from the epoch-100 held-out predictions. Recall is a health statistic, not grounding evidence.

The output below is intentionally not interpreted in advance. After execution,
its review must record: literal observation → strongest alternative explanation
→ discriminating test → limited conclusion → next question.


In [ ]:
# ALT: Four aligned dot plots showing raw-score spread, label separation, balanced accuracy, and positive recall for every FunnyBird concept.
def balanced_accuracy(y, pred):
    y=np.asarray(y).astype(int); pred=np.asarray(pred).astype(int)
    tpr=(pred[y==1]==1).mean() if (y==1).any() else np.nan
    tnr=(pred[y==0]==0).mean() if (y==0).any() else np.nan
    return np.nanmean([tpr,tnr])

rows=[]
for j,name in enumerate(CONCEPT_NAMES):
    z=z_saved[:,j]; c=c_saved[:,j].astype(int); pred=(z>0).astype(int)
    rows.append({"concept":name,"part":CONCEPT_PART[name],
                 "spread":np.quantile(z,.95)-np.quantile(z,.05),
                 "label_separation":np.median(z[c==1])-np.median(z[c==0]),
                 "balanced_accuracy":balanced_accuracy(c,pred),
                 "positive_recall":pred[c==1].mean(),
                 "n_positive":int(c.sum()),"n_negative":int((c==0).sum())})
HEALTH=pd.DataFrame(rows).sort_values(["part","concept"])
y_true=np.asarray(saved["y"]).reshape(-1).astype(int)
y_scores=np.asarray(saved["y_preds"])
if y_scores.ndim>2: y_scores=y_scores.reshape(len(y_scores),-1)
task_accuracy=float((y_scores.argmax(1)==y_true).mean())
concept_accuracy=float(((z_saved>0)==c_saved).mean())
display(pd.DataFrame([{"images":len(y_true),"species":len(np.unique(y_true)),
                      "task_accuracy":task_accuracy,"concept_accuracy":concept_accuracy}]).round(4))
display(HEALTH.round(3))
metrics=["spread","label_separation","balanced_accuracy","positive_recall"]
fig,axes=plt.subplots(1,4,figsize=(15,max(5,.24*len(HEALTH))),sharey=True)
y=np.arange(len(HEALTH))
for ax,m in zip(axes,metrics):
    ax.scatter(HEALTH[m],y,c=HEALTH.part.map(COLORS),s=24)
    ax.set_xlabel(m.replace("_"," "))
    if m in ["label_separation"]: ax.axvline(0,color="black",lw=.8)
    if m in ["balanced_accuracy","positive_recall"]: ax.axvline(.5,color="gray",ls="--",lw=.8)
axes[0].set_yticks(y); axes[0].set_yticklabels(HEALTH.concept,fontsize=7)
axes[0].invert_yaxis(); fig.suptitle("Figure 1 · Exact-concept model-health guard")
plt.tight_layout(); plt.show()


### Review record for Figure 1

- **Literal observation:** _Complete only after displaying this figure in chat._
- **Strongest alternative explanation:** _Pending visual review._
- **Discriminating test:** _Pending visual review._
- **Limited conclusion:** `INCOMPLETE — figure not yet reviewed`.
- **Next question:** _Complete after the limited conclusion is fixed._


## 2 · Did the renderer change only the intended part?

**Question.** Did the renderer change only the intended part?

**Variables and prediction.** Inspect the semantic preflight and original/swap/delete/part-map examples for all five parts. A valid intervention visibly changes the target part, preserves the rest of the scene, and has nonzero target-mask pixels.

**Method.** Use artifacts from the accepted fixed-render root before reading any model response.

The output below is intentionally not interpreted in advance. After execution,
its review must record: literal observation → strongest alternative explanation
→ discriminating test → limited conclusion → next question.


In [ ]:
# ALT: Complete FunnyBird intervention audit showing original, swapped, deleted, and part-map images for tail, wing, beak, foot, and eye.
ROOT = SWAP.parent
preflight_candidates=[ROOT/"renderer_preflight"/"renderer_semantic_preflight.png",
                      CURATED/"swap_fixed_v2_attempt2"/"renderer_preflight"/"renderer_semantic_preflight.png"]
preflight=next((p for p in preflight_candidates if p.exists()),preflight_candidates[0])
example_candidates=[ROOT/"examples",CURATED/"swap_fixed_v2_attempt2"/"examples"]
examples=next((p for p in example_candidates if p.is_dir()),example_candidates[0])
if preflight.exists():
    display(DisplayImage(filename=str(preflight)))
else:
    print("preflight sheet not stored beside CSV; use accepted job-3330289 audit")
from PIL import Image
tags=["orig","swap","delete","swap_partmap"]
fig,axes=plt.subplots(len(ORDER),len(tags),figsize=(12,13))
for r,part in enumerate(ORDER):
    for c,tag in enumerate(tags):
        ax=axes[r,c]; files=sorted(examples.glob(f"{part}_*_{tag}.png"))
        if files: ax.imshow(Image.open(files[0]).convert("RGB"))
        else: ax.text(.5,.5,"missing",ha="center",va="center")
        ax.set_title(f"{part} · {tag}"); ax.axis("off")
fig.suptitle("Figure 2 · Complete intervention audit: original, replacement, deletion, and target mask")
plt.tight_layout(); plt.show()


### Review record for Figure 2

- **Literal observation:** _Complete only after displaying this figure in chat._
- **Strongest alternative explanation:** _Pending visual review._
- **Discriminating test:** _Pending visual review._
- **Limited conclusion:** `INCOMPLETE — figure not yet reviewed`.
- **Next question:** _Complete after the limited conclusion is fixed._


## 3 · Did the inserted pixels move the comparison toward the donor?

**Question.** Did the inserted pixels move the comparison toward the donor?

**Variables and prediction.** `response_delta = (z_donor-z_source)_cf - (z_donor-z_source)_orig`. Values above zero mean that replacement pixels moved the model toward the donor concept.

**Method.** Plot the complete distribution for every part and report the positive-response rate.

The output below is intentionally not interpreted in advance. After execution,
its review must record: literal observation → strongest alternative explanation
→ discriminating test → limited conclusion → next question.


In [ ]:
# ALT: FunnyBird response-delta distributions and positive donor-response rates for all five parts.
fig,axes=plt.subplots(1,2,figsize=(12,4.2))
vals=[S.loc[S.part==p,"response_delta"].dropna() for p in ORDER]
bp=axes[0].boxplot(vals,tick_labels=ORDER,showfliers=False,whis=(5,95),patch_artist=True)
for box,p in zip(bp["boxes"],ORDER): box.set_facecolor(COLORS[p]); box.set_alpha(.55)
axes[0].axhline(0,color="black",lw=1); axes[0].set_ylabel("response_delta (raw logit units)")
axes[0].set_title("A · Distribution of donorward movement")
rate=S.groupby("part").response_delta.apply(lambda x:(x>0).mean()).reindex(ORDER)
axes[1].bar(rate.index,rate.values,color=[COLORS[p] for p in rate.index])
axes[1].axhline(.5,color="gray",ls="--"); axes[1].set_ylim(0,1)
axes[1].set_ylabel("fraction with response_delta > 0"); axes[1].set_title("B · Positive donor-response rate")
fig.suptitle("Figure 3 · Does the replacement produce the predicted within-image response?")
plt.tight_layout(); plt.show(); display(rate.rename("positive_response_rate").to_frame().round(3))


### Review record for Figure 3

- **Literal observation:** _Complete only after displaying this figure in chat._
- **Strongest alternative explanation:** _Pending visual review._
- **Discriminating test:** _Pending visual review._
- **Limited conclusion:** `INCOMPLETE — figure not yet reviewed`.
- **Next question:** _Complete after the limited conclusion is fixed._


## 4 · After responding, does the donor finish above the old source?

**Question.** After responding, does the donor finish above the old source?

**Variables and prediction.** The final margin is `m_cf=z_donor,cf-z_source,cf`. The primary event is `response_delta>0` with `m_cf<0`. A lower-right quadrant point means the inserted pixels had an effect but the old source still wins.

**Method.** Show final-margin distributions and the joint response/margin plane for every part.

The output below is intentionally not interpreted in advance. After execution,
its review must record: literal observation → strongest alternative explanation
→ discriminating test → limited conclusion → next question.


In [ ]:
# ALT: Final donor-minus-source margin distributions and joint response-delta versus final-margin plot for all FunnyBird parts.
fig,axes=plt.subplots(1,2,figsize=(14,4.8))
vals=[S.loc[S.part==p,"margin"].dropna() for p in ORDER]
bp=axes[0].boxplot(vals,tick_labels=ORDER,showfliers=False,whis=(5,95),patch_artist=True)
for box,p in zip(bp["boxes"],ORDER): box.set_facecolor(COLORS[p]); box.set_alpha(.55)
axes[0].axhline(0,color="black",lw=1); axes[0].set_ylabel("final margin m_cf (donor − source)")
axes[0].set_title("A · Final donor-minus-source margin")
for p in ORDER:
    d=S[S.part==p]
    axes[1].scatter(d.response_delta,d.margin,s=10,alpha=.22,color=COLORS[p],label=p)
axes[1].axvline(0,color="black",lw=1); axes[1].axhline(0,color="black",lw=1)
axes[1].set_xlabel("response_delta"); axes[1].set_ylabel("final margin m_cf")
axes[1].set_title("B · Lower-right = responds, but old source still wins")
axes[1].legend(ncol=5,fontsize=8)
fig.suptitle("Figure 4 · Controlled FunnyBird backwash predicate")
plt.tight_layout(); plt.show()
summary=S.groupby("part").agg(n=("margin","size"),median_response=("response_delta","median"),
    median_final_margin=("margin","median"),positive_response_rate=("response_delta",lambda x:(x>0).mean()),
    responded_but_source_wins_rate=("responded_but_source_wins","mean")).reindex(ORDER)
display(summary.round(3))


### Review record for Figure 4

- **Literal observation:** _Complete only after displaying this figure in chat._
- **Strongest alternative explanation:** _Pending visual review._
- **Discriminating test:** _Pending visual review._
- **Limited conclusion:** `INCOMPLETE — figure not yet reviewed`.
- **Next question:** _Complete after the limited conclusion is fixed._


## 5 · Could opposite swap directions create the result?

**Question.** Could opposite swap directions create the result?

**Variables and prediction.** Compare forward and backward rates of `response_delta>0 and final margin<0`, together with median margins. A genuine part pattern should appear in both directions rather than cancel when pooled.

**Method.** Keep directions separate and show their denominators.

The output below is intentionally not interpreted in advance. After execution,
its review must record: literal observation → strongest alternative explanation
→ discriminating test → limited conclusion → next question.


In [ ]:
# ALT: Forward and backward FunnyBird rates where the donor changes the margin but the old source remains larger, alongside final margins for every part.
D=(S.groupby(["part","direction"]).agg(n=("margin","size"),median_margin=("margin","median"),
     responded_but_source_wins_rate=("responded_but_source_wins","mean")).reset_index())
fig,axes=plt.subplots(1,2,figsize=(12,4))
for direction,marker in [("fwd","o"),("bwd","s")]:
    d=D[D.direction==direction].set_index("part").reindex(ORDER)
    axes[0].plot(ORDER,d.responded_but_source_wins_rate,marker=marker,label=direction)
    axes[1].plot(ORDER,d.median_margin,marker=marker,label=direction)
axes[0].set_ylim(0,1); axes[0].set_ylabel("fraction: donorward response, but source still wins")
axes[1].axhline(0,color="black",lw=.8); axes[1].set_ylabel("median final margin")
axes[0].legend(); axes[1].legend(); fig.suptitle("Figure 5 · Forward and backward directions")
plt.tight_layout(); plt.show(); display(D.round(3))


### Review record for Figure 5

- **Literal observation:** _Complete only after displaying this figure in chat._
- **Strongest alternative explanation:** _Pending visual review._
- **Discriminating test:** _Pending visual review._
- **Limited conclusion:** `INCOMPLETE — figure not yet reviewed`.
- **Next question:** _Complete after the limited conclusion is fixed._


## 6 · How much of the result is associated with target visibility?

**Question.** How much of the result is associated with target visibility?

**Variables and prediction.** Use `pixel_count_cf` from the exact swapped-part map and the same final-margin and `response_delta>0, margin<0` definition. If visibility is sufficient, highly visible replacements should remove the part gap; a remaining gap requires another explanation.

**Method.** Use declared bins and print the number of swap rows in every bin.

The output below is intentionally not interpreted in advance. After execution,
its review must record: literal observation → strongest alternative explanation
→ discriminating test → limited conclusion → next question.


In [ ]:
# ALT: FunnyBird final margin and responded-but-source-still-wins rate across exact swapped-part visibility bins for all parts.
if "pixel_count_cf" not in S: raise RuntimeError("fixed swap CSV lacks pixel_count_cf")
bins=[0,20,50,100,200,500,np.inf]; labels=["0–19","20–49","50–99","100–199","200–499","500+"]
V=S.copy(); V["visibility_bin"]=pd.cut(V.pixel_count_cf,bins=bins,labels=labels,right=False)
T=V.groupby(["part","visibility_bin"],observed=True).agg(
    n=("margin","size"),median_margin=("margin","median"),responded_but_source_wins_rate=("responded_but_source_wins","mean")).reset_index()
fig,axes=plt.subplots(1,2,figsize=(14,4.5))
for p in ORDER:
    d=T[T.part==p].set_index("visibility_bin").reindex(labels)
    axes[0].plot(labels,d.median_margin,"o-",label=p,color=COLORS[p])
    axes[1].plot(labels,d.responded_but_source_wins_rate,"o-",label=p,color=COLORS[p])
axes[0].axhline(0,color="black",lw=.8); axes[0].set_ylabel("median final margin")
axes[1].set_ylim(0,1); axes[1].set_ylabel("fraction: donorward response, but source still wins")
for ax in axes: ax.tick_params(axis="x",rotation=45); ax.legend(fontsize=8,ncol=2)
fig.suptitle("Figure 6 · Same-render visibility analysis")
plt.tight_layout(); plt.show(); display(T.round(3))


### Review record for Figure 6

- **Literal observation:** _Complete only after displaying this figure in chat._
- **Strongest alternative explanation:** _Pending visual review._
- **Discriminating test:** _Pending visual review._
- **Limited conclusion:** `INCOMPLETE — figure not yet reviewed`.
- **Next question:** _Complete after the limited conclusion is fixed._


## 6b · How often did the original training label conflict with visible part evidence?

**Question.** How often did the original training label conflict with visible part evidence?

**Variables and prediction.** Compare the standard and visibility-aware training records for the same images; count positive concept labels changed to zero within each part group. A large conflict count identifies a plausible training signal that can reward contextual prediction, but its causal effect belongs to notebook 03rl.

**Method.** Require identical ordered image/class records and allow only `attribute_label` to differ.

The output below is intentionally not interpreted in advance. After execution,
its review must record: literal observation → strongest alternative explanation
→ discriminating test → limited conclusion → next question.


In [ ]:
# ALT: FunnyBird training-image counts whose positive part-concept labels change under the matched visibility-aware relabeling rule.
import pickle
std_path=CURATED/"funnybirds_processed_trainval"/"train.pkl"
rl_path=CURATED/"funnybirds_processed_rl_trainval"/"train.pkl"
if not (std_path.exists() and rl_path.exists()):
    print("INCOMPLETE: matched standard/RLv2 training records are not both present")
else:
    std=pickle.loads(std_path.read_bytes()); rl=pickle.loads(rl_path.read_bytes())
    if len(std)!=len(rl): raise RuntimeError("standard/RLv2 train lengths differ")
    changes={p:0 for p in ORDER}; image_changes={p:0 for p in ORDER}
    for a,b in zip(std,rl):
        for key in a:
            if key=="attribute_label": continue
            av,bv=a[key],b[key]
            equal=np.array_equal(np.asarray(av),np.asarray(bv)) if isinstance(av,(list,tuple,np.ndarray)) else av==bv
            if not bool(equal): raise RuntimeError(f"non-label record field differs: {key}")
        ca=np.asarray(a["attribute_label"]); cb=np.asarray(b["attribute_label"])
        for p,(lo,hi) in SPANS.items():
            n=int(((ca[lo:hi]==1)&(cb[lo:hi]==0)).sum()); changes[p]+=n; image_changes[p]+=int(n>0)
    CONFLICT=pd.DataFrame({"changed_positive_labels":changes,"images_with_change":image_changes}).reindex(ORDER)
    fig,ax=plt.subplots(figsize=(8,4)); ax.bar(CONFLICT.index,CONFLICT.images_with_change,color=[COLORS[p] for p in CONFLICT.index])
    ax.set_ylabel("training images with ≥1 positive label removed")
    ax.set_title("Figure 6b · Original label/visibility conflict by part")
    plt.tight_layout(); plt.show(); display(CONFLICT)


### Review record for Figure 6b

- **Literal observation:** _Complete only after displaying this figure in chat._
- **Strongest alternative explanation:** _Pending visual review._
- **Discriminating test:** _Pending visual review._
- **Limited conclusion:** `INCOMPLETE — figure not yet reviewed`.
- **Next question:** _Complete after the limited conclusion is fixed._


## 7 · Do exact source and donor values explain the failures?

**Question.** Do exact source and donor values explain the failures?

**Variables and prediction.** For every part, compare the inserted donor value with the concept value that has the largest post-swap raw score. A clean diagonal means exact visual values are distinguished; recurring bright columns indicate default answers.

**Method.** Display all parts and all values with row-normalized counts.

The output below is intentionally not interpreted in advance. After execution,
its review must record: literal observation → strongest alternative explanation
→ discriminating test → limited conclusion → next question.


In [ ]:
# ALT: Five row-normalized confusion matrices comparing inserted and highest-scoring FunnyBird part values.
available=[p for p in ORDER if any(c.startswith(f"z_cf_{p}_") for c in S.columns)]
if set(available)!=set(ORDER): raise RuntimeError(f"missing all-part post-swap concept logits: have {available}")
fig,axes=plt.subplots(1,5,figsize=(17,3.5))
diag={}
for ax,p in zip(axes,ORDER):
    cols=sorted([c for c in S if c.startswith(f"z_cf_{p}_")],key=lambda x:int(x.rsplit("_",1)[1]))
    d=S[S.part==p].dropna(subset=cols); donor=d.var_donor.astype(int).to_numpy(); pred=d[cols].to_numpy().argmax(1)
    M=np.zeros((len(cols),len(cols)))
    for a,b in zip(donor,pred):
        if 0<=a<len(cols): M[a,b]+=1
    M=M/np.maximum(M.sum(1,keepdims=True),1); diag[p]=(donor==pred).mean()
    im=ax.imshow(M,vmin=0,vmax=1,cmap="magma"); ax.set_title(f"{p}\ndiagonal={diag[p]:.2f}")
    ax.set_xlabel("highest-scoring value"); ax.set_ylabel("inserted value")
fig.colorbar(im,ax=axes,fraction=.015); fig.suptitle("Figure 7 · Exact-value attribution after controlled replacement")
plt.tight_layout(); plt.show(); display(pd.Series(diag,name="diagonal_rate").to_frame().round(3))


### Review record for Figure 7

- **Literal observation:** _Complete only after displaying this figure in chat._
- **Strongest alternative explanation:** _Pending visual review._
- **Discriminating test:** _Pending visual review._
- **Limited conclusion:** `INCOMPLETE — figure not yet reviewed`.
- **Next question:** _Complete after the limited conclusion is fixed._


## 7b · Are difficult values simply rare or drawn from a larger alternative set?

**Question.** Are difficult values simply rare or drawn from a larger alternative set?

**Variables and prediction.** For every donor value, compare its source-species support with the rate where `response_delta>0` but the final margin remains negative; also report the total number of alternatives for its part. An association supports frequency or choice-set difficulty, but five part-level counts cannot establish a stable correlation.

**Method.** Label every exact value and show its number of swap rows.

The output below is intentionally not interpreted in advance. After execution,
its review must record: literal observation → strongest alternative explanation
→ discriminating test → limited conclusion → next question.


In [ ]:
# ALT: Labelled FunnyBird donor-value plot of species support versus the rate where donor pixels move the margin but the old source remains larger.
VS=(S.groupby(["part","var_donor"]).agg(n_rows=("margin","size"),species_support=("sid_donor","nunique"),
     responded_but_source_wins_rate=("responded_but_source_wins","mean"),median_margin=("margin","median")).reset_index())
VS["alternatives_in_part"]=VS.part.map({p:hi-lo for p,(lo,hi) in SPANS.items()})
fig,ax=plt.subplots(figsize=(9,6))
for p,d in VS.groupby("part"):
    ax.scatter(d.species_support,d.responded_but_source_wins_rate,s=35,color=COLORS[p],label=p)
    for r in d.itertuples(): ax.annotate(f"{p}_{int(r.var_donor)}",(r.species_support,r.responded_but_source_wins_rate),fontsize=6,xytext=(3,3),textcoords="offset points")
ax.set_xlabel("source species carrying donor value"); ax.set_ylabel("fraction: donorward response, but source still wins")
ax.set_ylim(-.02,1.02); ax.legend(); ax.set_title("Figure 7b · Exact-value support versus controlled backwash events")
plt.tight_layout(); plt.show(); display(VS.round(3))


### Review record for Figure 7b

- **Literal observation:** _Complete only after displaying this figure in chat._
- **Strongest alternative explanation:** _Pending visual review._
- **Discriminating test:** _Pending visual review._
- **Limited conclusion:** `INCOMPLETE — figure not yet reviewed`.
- **Next question:** _Complete after the limited conclusion is fixed._


## 8 · Does source species organize the remaining error after exact values?

**Question.** Does source species organize the remaining error after exact values?

**Variables and prediction.** Subtract the mean margin for each `(part, source value, donor value)` combination, then summarize the residual by source species. Persistent species differences support an additional unchanged-body/species association, but remain observational.

**Method.** Show every part and require at least five rows per displayed species estimate.

The output below is intentionally not interpreted in advance. After execution,
its review must record: literal observation → strongest alternative explanation
→ discriminating test → limited conclusion → next question.


In [ ]:
# ALT: Per-source-species FunnyBird margin residuals after controlling exact source and donor values, shown for all parts.
R=S.copy(); R["value_pair_mean"]=R.groupby(["part","var_src","var_donor"]).margin.transform("mean")
R["margin_after_value_pair"]=R.margin-R.value_pair_mean
SP=(R.groupby(["part","sid_src"]).agg(n=("margin","size"),residual=("margin_after_value_pair","mean"))
      .reset_index().query("n>=5"))
fig,axes=plt.subplots(1,5,figsize=(18,4),sharey=True)
for ax,p in zip(axes,ORDER):
    d=SP[SP.part==p].sort_values("residual")
    ax.scatter(np.arange(len(d)),d.residual,color=COLORS[p],s=18)
    ax.axhline(0,color="black",lw=.8); ax.set_title(f"{p} (n species={len(d)})")
    ax.set_xlabel("source species, sorted")
axes[0].set_ylabel("mean margin residual after exact value pair")
fig.suptitle("Figure 8 · Source-species residual after exact source/donor values")
plt.tight_layout(); plt.show(); display(SP.groupby("part").residual.agg(["min","median","max","std","count"]).round(3))


### Review record for Figure 8

- **Literal observation:** _Complete only after displaying this figure in chat._
- **Strongest alternative explanation:** _Pending visual review._
- **Discriminating test:** _Pending visual review._
- **Limited conclusion:** `INCOMPLETE — figure not yet reviewed`.
- **Next question:** _Complete after the limited conclusion is fixed._


## 8b · How much species identity is recoverable from the learned concept vector?

**Question.** How much species identity is recoverable from the learned concept vector?

**Variables and prediction.** Train a held-out linear species probe on the full raw-logit vector and on each part block separately. Accuracy above the 1/50 chance level shows stored species information; it does not identify the pixels responsible or prove backward causal flow.

**Method.** Use one fixed stratified 70/30 split of the held-out prediction population.

The output below is intentionally not interpreted in advance. After execution,
its review must record: literal observation → strongest alternative explanation
→ discriminating test → limited conclusion → next question.


In [ ]:
# ALT: Held-out FunnyBird species-decoding accuracy from the complete raw concept vector and each individual part block.
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
y_saved=np.asarray(saved["y"]).reshape(-1).astype(int)
idx=np.arange(len(y_saved)); tr,te=train_test_split(idx,test_size=.30,random_state=20260803,stratify=y_saved)
blocks={"complete z":np.arange(z_saved.shape[1])}
blocks.update({p:np.arange(lo,hi) for p,(lo,hi) in SPANS.items()})
probe=[]
for name,cols in blocks.items():
    model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=3000,C=1.0,random_state=20260803))
    model.fit(z_saved[tr][:,cols],y_saved[tr]); probe.append({"block":name,"species_accuracy":accuracy_score(y_saved[te],model.predict(z_saved[te][:,cols])),"dimensions":len(cols)})
PROBE=pd.DataFrame(probe)
fig,ax=plt.subplots(figsize=(8,4)); ax.bar(PROBE.block,PROBE.species_accuracy,color=["#333333"]+[COLORS.get(x,"#999999") for x in PROBE.block.iloc[1:]])
ax.axhline(1/len(np.unique(y_saved)),color="black",ls="--",label="chance = 1/50")
ax.set_ylim(0,1); ax.set_ylabel("held-out species accuracy"); ax.set_title("Figure 8b · Species decoded from learned concept representations")
ax.legend(); plt.tight_layout(); plt.show(); display(PROBE.round(3))


### Review record for Figure 8b

- **Literal observation:** _Complete only after displaying this figure in chat._
- **Strongest alternative explanation:** _Pending visual review._
- **Discriminating test:** _Pending visual review._
- **Limited conclusion:** `INCOMPLETE — figure not yet reviewed`.
- **Next question:** _Complete after the limited conclusion is fixed._


## 9 · How much does each observed block account for?

**Question.** How much does each observed block account for?

**Variables and prediction.** Predict the raw final margin on held-out render IDs using progressively richer categorical blocks. Lower held-out error means the added block organizes the outcome; remaining error is the measured residual.

**Method.** Use stable five-fold image-level splits, training-fold group means with shrinkage, and no RLv2 variables.

The output below is intentionally not interpreted in advance. After execution,
its review must record: literal observation → strongest alternative explanation
→ discriminating test → limited conclusion → next question.


In [ ]:
# ALT: Held-out final-margin prediction error after adding FunnyBird visibility, exact values, and source species sequentially.
import hashlib
A=S.copy(); A["vis_bin"]=pd.cut(A.pixel_count_cf,[-1,19,49,99,199,499,np.inf],labels=False)
unit=(A["render_id"].astype(str) if "render_id" in A else
      A.get("li",pd.Series(np.arange(len(A)),index=A.index)).astype(str))
A["fold"]=unit.map(lambda x:int(hashlib.sha1(x.encode()).hexdigest(),16)%5)
stages=[("part only",["part"]),("+ visibility",["part","vis_bin"]),
        ("+ exact values",["part","vis_bin","var_src","var_donor"]),
        ("+ source species",["part","vis_bin","var_src","var_donor","sid_src"])]
rows=[]
for stage,cols in stages:
    pred=pd.Series(index=A.index,dtype=float)
    for fold in range(5):
        tr=A[A.fold!=fold]; te=A[A.fold==fold]
        prior=tr.margin.mean(); stats=tr.groupby(cols).margin.agg(["mean","count"]).reset_index()
        stats["estimate"]=(stats["mean"]*stats["count"]+prior*10)/(stats["count"]+10)
        joined=te[cols].merge(stats[cols+["estimate"]],on=cols,how="left")
        pred.loc[te.index]=joined.estimate.fillna(prior).to_numpy()
    rows.append({"stage":stage,"rmse":float(np.sqrt(np.mean((A.margin-pred)**2))),
                 "mae":float(np.mean(np.abs(A.margin-pred)))})
ACCOUNT=pd.DataFrame(rows)
fig,ax=plt.subplots(figsize=(8,4)); ax.plot(ACCOUNT.stage,ACCOUNT.rmse,"o-",color="#0072B2")
ax.set_ylabel("held-out RMSE of final margin"); ax.tick_params(axis="x",rotation=25)
ax.set_title("Figure 9 · Sequential descriptive accounting on identical swap rows")
plt.tight_layout(); plt.show(); display(ACCOUNT.round(3))


### Review record for Figure 9

- **Literal observation:** _Complete only after displaying this figure in chat._
- **Strongest alternative explanation:** _Pending visual review._
- **Discriminating test:** _Pending visual review._
- **Limited conclusion:** `INCOMPLETE — figure not yet reviewed`.
- **Next question:** _Complete after the limited conclusion is fixed._


## 10 · Does the concept-layer error materially alter species prediction?

**Question.** Does the concept-layer error materially alter species prediction?

**Variables and prediction.** Relate final concept margin to the model's donor-species probability, which is a different downstream quantity. A small downstream change would limit the harm to explanation reliability rather than widespread class failure.

**Method.** Use independent final-margin bins and print bin counts.

The output below is intentionally not interpreted in advance. After execution,
its review must record: literal observation → strongest alternative explanation
→ discriminating test → limited conclusion → next question.


In [ ]:
# ALT: Binned relationship between FunnyBird final concept margin and downstream donor-species probability.
prob_col=next((c for c in ["p_cf_donor","p_donor_cf","donor_species_prob"] if c in S),None)
if prob_col is None:
    print("INCOMPLETE: swap CSV has no donor-species probability column")
else:
    D=S.copy(); D["margin_bin"]=pd.qcut(D.margin,10,duplicates="drop")
    Q=D.groupby("margin_bin",observed=True).agg(n=(prob_col,"size"),mean_margin=("margin","mean"),mean_donor_species_prob=(prob_col,"mean")).reset_index()
    fig,ax=plt.subplots(figsize=(7,4)); ax.plot(Q.mean_margin,Q.mean_donor_species_prob,"o-")
    for r in Q.itertuples(): ax.annotate(f"n={r.n}",(r.mean_margin,r.mean_donor_species_prob),fontsize=7)
    ax.axvline(0,color="black",lw=.8); ax.set_xlabel("mean final concept margin in bin")
    ax.set_ylabel("mean donor-species probability"); ax.set_title("Figure 10 · Downstream consequence of the concept margin")
    plt.tight_layout(); plt.show(); display(Q.round(3))


### Review record for Figure 10

- **Literal observation:** _Complete only after displaying this figure in chat._
- **Strongest alternative explanation:** _Pending visual review._
- **Discriminating test:** _Pending visual review._
- **Limited conclusion:** `INCOMPLETE — figure not yet reviewed`.
- **Next question:** _Complete after the limited conclusion is fixed._


## 11 · Standard-CBM evidence ledger

Complete this table only after Figures 1–10 have been displayed and reviewed.

| Predicate or explanation | Direct measurement | Status after review |
|---|---|---|
| model outputs are usable | Figure 1 | `INCOMPLETE` |
| interventions are valid | Figure 2 | `INCOMPLETE` |
| inserted pixels cause donorward movement | Figure 3 | `INCOMPLETE` |
| old source can remain stronger after that movement | Figure 4 | `INCOMPLETE` |
| direction artifact excluded | Figure 5 | `INCOMPLETE` |
| visibility contribution | Figure 6 | `INCOMPLETE` |
| training label/mask conflict measured | Figure 6b | `INCOMPLETE` |
| exact-value contribution | Figure 7 | `INCOMPLETE` |
| exact-value support contribution | Figure 7b | `INCOMPLETE` |
| source-species residual | Figure 8 | `INCOMPLETE` |
| species information in learned representation | Figure 8b | `INCOMPLETE` |
| sequential descriptive accounting | Figure 9 | `INCOMPLETE` |
| downstream class consequence | Figure 10 | `INCOMPLETE` |

**Next report question.** Notebook 03 asks whether the MCBM minimality
penalty changes these same accepted quantities. It must not replace the
standard-CBM result established here.


# Methods appendix · measurements not used in the main claim

The reciprocal mask-deletion and randomized-patch experiments are retained
as method-development history. They did not reproduce the clean FunnyBird
control sufficiently to transfer their causal interpretation to CUB.

- reciprocal mask deletion: `METHOD NOT CALIBRATED FOR CROSS-DATASET CAUSAL COMPARISON`;
- randomized patch V1/V2: local pixel response was measurable in selected
  examples, but the all-part control was not calibrated and wing coverage was
  inadequate;
- none of these outcomes invalidates the validated renderer swap above.

Full artifacts and scripts remain under `analysis/paired_mask_deletion.py`,
`analysis/randomized_patch_masking.py`, and their output directories. They
are not rerun by this notebook.


# Provenance appendix

Record after execution: Git commit, checkpoint path, prediction path, swap
CSV path, fixed-render audit path, row counts, seeds, and exclusions. A
stale HTML is not synchronized evidence.
